# Phase 2 with Workers Bindings (Puppeteer)

This notebook implements the Cloudflare Workers Bindings approach for faster crawling:
- **10 concurrent browsers** (stable limit for Workers Paid Plan)
- **Session reuse** - eliminates cold start latency
- **Direct Puppeteer control** - no API rate limits
- **~10x faster** than REST API approach

## Performance Comparison

| Metric | REST API | Workers Binding |
|--------|----------|-----------------|
| Concurrency | 20 | 10 |
| Speed | ~4 URLs/min | **~40 URLs/min** |
| Estimated time for 13,849 URLs | ~58 hours | **~5.8 hours** |

## Prerequisites

1. Worker must be deployed via wrangler:
   ```bash
   cd worker && wrangler deploy
   ```

2. Worker URL: `https://sgcarmart-browser-worker.npua271.workers.dev`

## 1. Setup

In [1]:
import os
import json
import asyncio
import ssl
import urllib.parse
import aiohttp
import re
from datetime import datetime
from pathlib import Path

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

CLOUDFLARE_ACCOUNT_ID = os.getenv("CLOUDFLARE_ACCOUNT_ID")
CLOUDFLARE_API_TOKEN = os.getenv("CLOUDFLARE_API_TOKEN")

# Worker URL (deployed via wrangler)
WORKER_URL = "https://sgcarmart-browser-worker.npua271.workers.dev"

print(f"Account ID: {CLOUDFLARE_ACCOUNT_ID[:8]}..." if CLOUDFLARE_ACCOUNT_ID else "Account ID not set")
print(f"API Token: {CLOUDFLARE_API_TOKEN[:8]}..." if CLOUDFLARE_API_TOKEN else "API Token not set")
print(f"Worker URL: {WORKER_URL}")

Account ID: 5decc977...
API Token: cfat_HAO...
Worker URL: https://sgcarmart-browser-worker.npua271.workers.dev


In [2]:
import aiohttp
import re
import random

class WorkersBindingCrawler:
    """Fast crawler using Cloudflare Workers with Browser Rendering binding.
    
    Extracts structured data from HTML - does NOT store HTML (99.9% storage savings).
    Includes retry logic with exponential backoff for rate limiting.
    """
    
    def __init__(self, worker_url: str, concurrency: int = 10, max_retries: int = 3):
        self.worker_url = worker_url
        self.concurrency = concurrency
        self.max_retries = max_retries
    
    # ============ Extraction Functions ============
    
    @staticmethod
    def extract_all_rsc_content(html: str) -> str:
        """Extract and combine all RSC push content from HTML."""
        push_pattern = r'self\.__next_f\.push\(\[(\d+),\s*"(.+?)"\]\)'
        push_matches = re.findall(push_pattern, html, re.DOTALL)
        
        all_content = ""
        for index_str, content in push_matches:
            try:
                unescaped = content.encode().decode('unicode_escape')
                all_content += unescaped
            except:
                all_content += content.replace('\\"', '"').replace('\\n', '\n')
        
        return all_content

    @staticmethod
    def parse_dollar_value(value):
        """Parse dollar-formatted values like '$$15,570 /yr' to numeric."""
        if value is None:
            return None
        cleaned = re.sub(r'[$,]', '', str(value))
        match = re.search(r'([\d]+)', cleaned)
        if match:
            return int(match.group(1))
        return None

    @staticmethod
    def find_data_object(content: str, start_pos: int) -> dict:
        """Find and parse a data object starting from a position."""
        data_start = content.find('"data":{', start_pos)
        if data_start < 0:
            return None
        
        brace_count = 0
        start_idx = data_start + 7
        i = start_idx
        while i < len(content):
            if content[i] == '{':
                brace_count += 1
            elif content[i] == '}':
                brace_count -= 1
                if brace_count == 0:
                    break
            i += 1
        
        json_str = content[start_idx:i+1]
        
        try:
            return json.loads(json_str)
        except:
            return None

    @classmethod
    def extract_detail_from_html(cls, html: str, url: str) -> dict:
        """Extract car detail fields from HTML.
        
        Returns structured data only - NO HTML storage (99.9% smaller).
        """
        detail_data = {"detail_page_url": url}
        
        all_content = cls.extract_all_rsc_content(html)
        
        # Find ucInfoDetailData
        search_pos = 0
        detail_data_found = None
        
        while True:
            detail_pos = all_content.find('"ucInfoDetailData"', search_pos)
            if detail_pos < 0:
                break
            
            success_pos = all_content.find('"success":true', detail_pos)
            if success_pos > 0 and success_pos < detail_pos + 100:
                data_obj = cls.find_data_object(all_content, success_pos)
                if data_obj and 'omv' in data_obj:
                    detail_data_found = data_obj
                    break
            
            search_pos = detail_pos + 1
        
        if detail_data_found:
            data = detail_data_found
            
            # Numeric fields that need parsing
            numeric_fields = {"depreciation", "coe", "road_tax", "omv", "arf", 
                              "mileage", "engine_cap", "power", "dereg_value"}
            
            # Field mapping from source to destination
            field_mapping = {
                "aid": "listing_id",
                "car_model": "car_model",
                "depreciation": "depreciation",
                "coe": "coe",
                "reg_date": "reg_date",
                "mileage": "mileage",
                "road_tax": "road_tax",
                "omv": "omv",
                "arf": "arf",
                "engine_cap": "engine_cap",
                "fuel_type": "fuel_type",
                "power": "power",
                "transmission": "transmission",
                "manufactured": "manufactured",
                "owners": "owners",
                "dereg_value": "dereg_value",
                "curb_weight": "curb_weight",
                "status": "status",
                "features": "features",
                "accessories": "accessories",
            }
            
            for src_field, dst_field in field_mapping.items():
                if src_field in data:
                    value = data[src_field]
                    if dst_field in numeric_fields:
                        detail_data[dst_field] = cls.parse_dollar_value(value)
                    else:
                        detail_data[dst_field] = value
            
            # Handle nested vehicle type
            if "type_of_vehicle" in data and isinstance(data["type_of_vehicle"], dict):
                detail_data["vehicle_type"] = data["type_of_vehicle"].get("text")
        
        # Find ucInfoDetailTopData for price
        search_pos = 0
        while True:
            top_pos = all_content.find('"ucInfoDetailTopData"', search_pos)
            if top_pos < 0:
                break
            
            success_pos = all_content.find('"success":true', top_pos)
            if success_pos > 0 and success_pos < top_pos + 100:
                top_data = cls.find_data_object(all_content, success_pos)
                if top_data and "price" in top_data:
                    detail_data["price"] = cls.parse_dollar_value(top_data["price"])
                    break
            
            search_pos = top_pos + 1
        
        return detail_data
    
    def _is_rate_limit_error(self, error_msg: str) -> bool:
        """Check if error is due to rate limiting."""
        return "429" in str(error_msg) or "Rate limit" in str(error_msg)
        
    async def crawl_url_with_retry(self, session, url: str) -> dict:
        """Crawl a single URL with retry logic and exponential backoff."""
        encoded_url = urllib.parse.quote(url, safe='')
        full_url = f"{self.worker_url}?url={encoded_url}"
        
        for attempt in range(self.max_retries):
            try:
                timeout = aiohttp.ClientTimeout(total=60)
                async with session.get(full_url, timeout=timeout) as response:
                    # Check HTTP status first
                    if response.status == 429:
                        # Rate limited - exponential backoff
                        wait_time = (2 ** attempt) + random.uniform(0, 1)
                        await asyncio.sleep(wait_time)
                        continue
                    
                    if response.status != 200:
                        text = await response.text()
                        if self._is_rate_limit_error(text):
                            wait_time = (2 ** attempt) + random.uniform(0, 1)
                            await asyncio.sleep(wait_time)
                            continue
                        return {"detail_page_url": url, "success": False, "error": f"HTTP {response.status}: {text[:200]}"}
                    
                    data = await response.json()
                    
                    # Check for worker-returned errors
                    if "error" in data:
                        if self._is_rate_limit_error(data["error"]) and attempt < self.max_retries - 1:
                            wait_time = (2 ** attempt) + random.uniform(0, 1)
                            await asyncio.sleep(wait_time)
                            continue
                        return {"detail_page_url": url, "success": False, "error": data["error"]}
                    
                    html = data.get("html", "")
                    
                    if not html:
                        return {"detail_page_url": url, "success": False, "error": "No HTML returned"}
                    
                    # Extract structured data (don't store HTML - saves 99.9% storage!)
                    detail = self.extract_detail_from_html(html, url)
                    detail["success"] = True
                    return detail
                    
            except asyncio.TimeoutError:
                if attempt < self.max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    await asyncio.sleep(wait_time)
                    continue
                return {"detail_page_url": url, "success": False, "error": "Timeout after 60s"}
            except Exception as e:
                if attempt < self.max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    await asyncio.sleep(wait_time)
                    continue
                return {"detail_page_url": url, "success": False, "error": f"{type(e).__name__}: {str(e)}"}
        
        return {"detail_page_url": url, "success": False, "error": f"Max retries ({self.max_retries}) exceeded"}
    
    async def batch_crawl(self, urls: list) -> list:
        """Batch crawl multiple URLs with high concurrency."""
        # Disable SSL verification (required for some environments)
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
        connector = aiohttp.TCPConnector(limit=self.concurrency, ssl=ssl_context)
        
        async with aiohttp.ClientSession(connector=connector) as session:
            tasks = [self.crawl_url_with_retry(session, url) for url in urls]
            results = await asyncio.gather(*tasks)
        return results


# Create crawler instance with concurrency 10 and 3 retries
crawler = WorkersBindingCrawler(WORKER_URL, concurrency=10, max_retries=3)
print("✓ WorkersBindingCrawler initialized")
print(f"  Worker URL: {WORKER_URL}")
print(f"  Concurrency: 10")
print(f"  Max retries: 3 (with exponential backoff)")
print(f"  Storage: Extracted data only (99.9% smaller than HTML)")

✓ WorkersBindingCrawler initialized
  Worker URL: https://sgcarmart-browser-worker.npua271.workers.dev
  Concurrency: 10
  Max retries: 3 (with exponential backoff)
  Storage: Extracted data only (99.9% smaller than HTML)


In [3]:
# Test the crawler with sample URLs
# Note: crawler is already created in cell-3

# Load URLs
with open("output/all_detail_urls.json") as f:
    all_urls = json.load(f)

# Test with 10 URLs (matches concurrency)
sample_urls = all_urls[:10]
print(f"Testing with {len(sample_urls)} URLs at concurrency=10...")
print(f"Worker URL: {WORKER_URL}")

start_time = datetime.now()
results = await crawler.batch_crawl(sample_urls)
total_time = (datetime.now() - start_time).total_seconds()

# Define successful BEFORE using it
successful = sum(1 for r in results if r.get("success"))
failed = len(results) - successful
success_rate = successful / len(sample_urls)

# Count extracted fields
all_fields = set()
for r in results:
    if r.get("success"):
        all_fields.update(k for k, v in r.items() if v is not None and k != "success")

print(f"\n=== Results ===")
print(f"Successful: {successful}/{len(sample_urls)}")
print(f"Failed: {failed}")
print(f"Total time: {total_time:.1f}s")
if total_time > 0:
    print(f"Speed: {successful/total_time*60:.1f} URLs/min")
print(f"Fields extracted: {sorted(all_fields)}")

# Show first few results
print(f"\n=== Sample Results (first 5) ===")
for i, r in enumerate(results[:5]):
    if r.get("success"):
        model = r.get('car_model', 'N/A')[:40]
        print(f"  {i+1}. ✓ {model}")
        if r.get('price'):
            print(f"     Price: ${r['price']:,}")
        if r.get('omv'):
            print(f"     OMV: {r['omv']:,}")
        field_count = len([k for k,v in r.items() if v is not None and k != 'success'])
        print(f"     Fields: {field_count}")
    else:
        url_short = r.get("detail_page_url", "")[:50]
        error = r.get('error', 'Unknown error')
        print(f"  {i+1}. ✗ {url_short}... - Error: {error}")
    print()

# Error breakdown (outside the for loop)
if failed > 0:
    print(f"\n=== Error Breakdown ===")
    error_types = {}
    for r in results:
        if not r.get("success"):
            error = r.get("error", "Unknown")
            if "Timeout" in error:
                error_cat = "Timeout"
            elif "HTTP" in error:
                error_cat = "HTTP Error"
            elif "No HTML" in error:
                error_cat = "No HTML"
            elif "Rate limit" in error or "429" in error:
                error_cat = "Rate Limit (429)"
            else:
                error_cat = "Other"
            error_types[error_cat] = error_types.get(error_cat, 0) + 1
    
    for error_type, count in sorted(error_types.items(), key=lambda x: -x[1]):
        print(f"  {error_type}: {count}")

Testing with 10 URLs at concurrency=10...
Worker URL: https://sgcarmart-browser-worker.npua271.workers.dev

=== Results ===
Successful: 10/10
Failed: 0
Total time: 18.5s
Speed: 32.4 URLs/min
Fields extracted: ['accessories', 'arf', 'car_model', 'coe', 'curb_weight', 'depreciation', 'dereg_value', 'detail_page_url', 'engine_cap', 'features', 'fuel_type', 'listing_id', 'manufactured', 'mileage', 'omv', 'owners', 'power', 'price', 'reg_date', 'road_tax', 'status', 'transmission', 'vehicle_type']

=== Sample Results (first 5) ===
  1. ✓ Lamborghini Gallardo LP560-4
     Price: $195,800
     OMV: 222,743
     Fields: 21

  2. ✓ Bentley Continental GT 4.0A V8
     Price: $228,800
     OMV: 206,137
     Fields: 23

  3. ✓ Honda Civic 1.6A VTi
     Price: $72,800
     OMV: 20,084
     Fields: 23

  4. ✓ Mercedes-Benz B-Class B180
     Price: $41,800
     OMV: 27,986
     Fields: 23

  5. ✓ Mercedes-Benz GLB-Class GLB200 AMG Line 
     Price: $131,800
     OMV: 38,946
     Fields: 23



## 3. Full Production Run

Run this section for full production crawling of all URLs.

In [ ]:
# Full production run (INCREMENTAL - skips already crawled URLs)
# Features: Retry logic, exponential backoff, batch delays

BATCH_SIZE = 50  # Process in batches to avoid memory issues
BATCH_DELAY = 2  # Seconds to wait between batches (avoids rate limiting)
OUTPUT_FILE = Path("output/workers_binding_results.json")
PARQUET_FILE = Path("output/full_detail_data.parquet")

# Load existing results to skip already crawled URLs
existing_urls = set()
all_results = []

if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE) as f:
        existing_results = json.load(f)
    existing_urls = {r.get("detail_page_url") for r in existing_results if r.get("detail_page_url")}
    all_results = existing_results
    print(f"Loaded {len(existing_urls)} already crawled URLs from previous runs")

# Filter to only new URLs
new_urls = [url for url in all_urls if url not in existing_urls]
total_urls = len(all_urls)
remaining = len(new_urls)

print(f"\nStarting incremental crawl...")
print(f"Total URLs: {total_urls}")
print(f"Already crawled: {len(existing_urls)}")
print(f"Remaining to crawl: {remaining}")
print(f"Batch size: {BATCH_SIZE}, Concurrency: 10")
print(f"Batch delay: {BATCH_DELAY}s (prevents rate limiting)")
print(f"Retry: 3 attempts with exponential backoff")

if remaining == 0:
    print("\n✅ All URLs already crawled! No new URLs to process.")
else:
    print(f"Estimated time: {remaining / 45:.1f} minutes (~{remaining / 45 / 60:.1f} hours)")
    print()
    
    start_total = datetime.now()
    failed_urls = []  # Track failed URLs for retry later
    
    for i in range(0, remaining, BATCH_SIZE):
        batch = new_urls[i:i+BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        total_batches = (remaining + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Batch {batch_num}/{total_batches}: Crawling {len(batch)} URLs...", end=" ")
        
        start = datetime.now()
        results = await crawler.batch_crawl(batch)
        elapsed = (datetime.now() - start).total_seconds()
        
        successful = sum(1 for r in results if r.get("success"))
        batch_failed = [r for r in results if not r.get("success")]
        
        print(f"Done: {successful}/{len(batch)} in {elapsed:.1f}s ({successful/elapsed*60:.0f} URLs/min)")
        
        if batch_failed:
            failed_urls.extend(batch_failed)
            # Show error breakdown for this batch
            errors = {}
            for r in batch_failed:
                err = r.get("error", "Unknown")[:50]
                errors[err] = errors.get(err, 0) + 1
            for err, count in sorted(errors.items(), key=lambda x: -x[1])[:2]:
                print(f"    ⚠️ {err}: {count}")
        
        all_results.extend(results)
        
        # Save checkpoint every 20 batches
        if batch_num % 20 == 0:
            with open(OUTPUT_FILE, "w") as f:
                json.dump(all_results, f, indent=2)
            print(f"  💾 Checkpoint saved: {len(all_results)} total results ({sum(1 for r in all_results if r.get('success'))} successful)")
        
        # Delay between batches to avoid rate limiting
        if batch_num < total_batches and BATCH_DELAY > 0:
            await asyncio.sleep(BATCH_DELAY)
    
    total_elapsed = (datetime.now() - start_total).total_seconds()

# Final save to JSON with pretty formatting
with open(OUTPUT_FILE, "w") as f:
    json.dump(all_results, f, indent=2)




Starting incremental crawl...
Total URLs: 14330
Already crawled: 0
Remaining to crawl: 14330
Batch size: 50, Concurrency: 10
Batch delay: 2s (prevents rate limiting)
Retry: 3 attempts with exponential backoff
Estimated time: 318.4 minutes (~5.3 hours)

Batch 1/287: Crawling 50 URLs... Done: 50/50 in 77.0s (39 URLs/min)
Batch 2/287: Crawling 50 URLs... Done: 50/50 in 73.4s (41 URLs/min)
Batch 3/287: Crawling 50 URLs... Done: 50/50 in 73.2s (41 URLs/min)
Batch 4/287: Crawling 50 URLs... Done: 49/50 in 96.3s (31 URLs/min)
    ⚠️ HTTP 500: {"error":"Navigation timeout of 30000 ms: 1
Batch 5/287: Crawling 50 URLs... Done: 50/50 in 77.0s (39 URLs/min)
Batch 6/287: Crawling 50 URLs... Done: 48/50 in 93.0s (31 URLs/min)
    ⚠️ HTTP 500: {"error":"Navigation timeout of 30000 ms: 2
Batch 7/287: Crawling 50 URLs... Done: 50/50 in 70.9s (42 URLs/min)
Batch 8/287: Crawling 50 URLs... Done: 50/50 in 80.4s (37 URLs/min)
Batch 9/287: Crawling 50 URLs... Done: 50/50 in 72.2s (42 URLs/min)
Batch 10/287

ComputeError: could not append value: "N.A." of type: str to the builder; make sure that all rows have the same schema or consider increasing `infer_schema_length`

it might also be that a value overflows the data-type's capacity

In [10]:
# Retry failed URLs
# Extracts failed URLs from all_results, retries them, and merges back

failed_records = [r for r in all_results if not r.get("success")]
failed_urls = [r["detail_page_url"] for r in failed_records]

print(f"Found {len(failed_urls)} failed URLs to retry")

# Error breakdown
error_types = {}
for r in failed_records:
    error = r.get("error", "Unknown")
    if "Timeout" in error:
        error_cat = "Timeout"
    elif "HTTP" in error:
        error_cat = "HTTP Error"
    elif "No HTML" in error:
        error_cat = "No HTML"
    elif "Rate limit" in error or "429" in error:
        error_cat = "Rate Limit (429)"
    else:
        error_cat = "Other"
    error_types[error_cat] = error_types.get(error_cat, 0) + 1

print("\nError breakdown:")
for err_type, count in sorted(error_types.items(), key=lambda x: -x[1]):
    print(f"  {err_type}: {count}")

if failed_urls:
    print(f"\nRetrying {len(failed_urls)} URLs with concurrency=10...")
    start_retry = datetime.now()
    retry_results = await crawler.batch_crawl(failed_urls)
    retry_elapsed = (datetime.now() - start_retry).total_seconds()
    
    retry_successful = sum(1 for r in retry_results if r.get("success"))
    print(f"Retry complete: {retry_successful}/{len(failed_urls)} successful in {retry_elapsed:.1f}s")
    
    # Replace failed records in all_results with retry results
    failed_url_set = set(failed_urls)
    all_results = [r for r in all_results if r.get("detail_page_url") not in failed_url_set]
    all_results.extend(retry_results)
    
    # Save updated results
    with open(OUTPUT_FILE, "w") as f:
        json.dump(all_results, f, indent=2)
    
    still_failed = sum(1 for r in retry_results if not r.get("success"))
    total_successful = sum(1 for r in all_results if r.get("success"))
    print(f"\nUpdated totals: {total_successful}/{len(all_results)} successful ({still_failed} still failing)")
    print(f"Saved to {OUTPUT_FILE}")
else:
    print("\nNo failed URLs to retry!")

Found 84 failed URLs to retry

Error breakdown:
  Timeout: 74
  Other: 8
  HTTP Error: 2

Retrying 84 URLs with concurrency=10...
Retry complete: 79/84 successful in 141.6s

Updated totals: 14325/14330 successful (5 still failing)
Saved to output/workers_binding_results.json


In [11]:
# Export to Parquet
import polars as pl

successful_results = [r for r in all_results if r.get("success")]

if successful_results:
    # Normalize sentinel values ("N.A.", "-", etc.) to None for consistent typing
    def clean_record(rec):
        cleaned = {}
        for k, v in rec.items():
            if k == "success":
                continue
            if isinstance(v, str) and v.strip().upper() in ("N.A.", "NA", "-", "N/A", "NIL"):
                cleaned[k] = None
            else:
                cleaned[k] = v
        return cleaned

    cleaned_results = [clean_record(r) for r in successful_results]
    df = pl.DataFrame(cleaned_results, infer_schema_length=None)
    df.write_parquet(PARQUET_FILE, compression="snappy")
    
    # Also save CSV
    df.write_csv(str(PARQUET_FILE).replace(".parquet", ".csv"))

total_successful = sum(1 for r in all_results if r.get("success"))
total_crawled = len(all_results)
print(f"\n{'='*60}")
print(f"✅ FINAL RESULTS")
print(f"{'='*60}")
print(f"Total records: {total_crawled}")
print(f"Successful: {total_successful} ({total_successful/total_crawled*100:.1f}%)")
print(f"Failed: {total_crawled - total_successful}")
if remaining > 0:
    print(f"New URLs crawled this run: {remaining}")
    print(f"Time for this run: {total_elapsed/60:.1f} minutes ({total_elapsed/3600:.2f} hours)")
    new_successful = sum(1 for r in all_results[-remaining:] if r.get('success'))
    if total_elapsed > 0:
        print(f"Speed: {new_successful/total_elapsed*60:.1f} URLs/min")
print(f"\n📁 Output files:")
print(f"   JSON:   {OUTPUT_FILE} ({Path(OUTPUT_FILE).stat().st_size / 1024 / 1024:.1f} MB)")
print(f"   Parquet: {PARQUET_FILE} ({Path(PARQUET_FILE).stat().st_size / 1024 / 1024:.1f} MB)")
print(f"   CSV:     {str(PARQUET_FILE).replace('.parquet', '.csv')}")
if successful_results:
    print(f"\n📊 DataFrame shape: {df.shape}")
    print(f"   Columns: {sorted(df.columns)}")


✅ FINAL RESULTS
Total records: 14330
Successful: 14325 (100.0%)
Failed: 5
New URLs crawled this run: 14330
Time for this run: 370.2 minutes (6.17 hours)
Speed: 38.7 URLs/min

📁 Output files:
   JSON:   output/workers_binding_results.json (12.3 MB)
   Parquet: output/full_detail_data.parquet (2.0 MB)
   CSV:     output/full_detail_data.csv

📊 DataFrame shape: (14325, 23)
   Columns: ['accessories', 'arf', 'car_model', 'coe', 'curb_weight', 'depreciation', 'dereg_value', 'detail_page_url', 'engine_cap', 'features', 'fuel_type', 'listing_id', 'manufactured', 'mileage', 'omv', 'owners', 'power', 'price', 'reg_date', 'road_tax', 'status', 'transmission', 'vehicle_type']


In [12]:
df.head()

detail_page_url,listing_id,car_model,depreciation,coe,reg_date,mileage,road_tax,omv,arf,engine_cap,fuel_type,power,transmission,manufactured,owners,dereg_value,curb_weight,status,features,accessories,vehicle_type,price
str,i64,str,i64,i64,str,i64,i64,i64,i64,i64,str,i64,str,i64,str,i64,str,str,str,str,str,i64
"""https://www.sgcarmart.com/used…",1488131,"""Lamborghini Gallardo LP560-4""",36950,57955,"""19-Jul-2011""",55000,8751,222743,222743,5204,"""Petrol""",412,"""Auto""",2011,"""More than 6""",30708,"""1,500 kg""","""Available for sale""",null,null,"""Sports Car""",195800
"""https://www.sgcarmart.com/used…",1464426,"""Bentley Continental GT 4.0A V8""",31960,115270,"""28-May-2013""",74202,5122,206137,206137,3993,"""Petrol""",373,"""Auto""",2013,"""6 Owners""",82521,"""2,295 kg""","""Available for sale""","""Powerful 4.0L V8 twin turbocha…","""Upgraded sound system, reverse…","""Sports Car""",228800
"""https://www.sgcarmart.com/used…",1478231,"""Honda Civic 1.6A VTi""",18590,26999,"""16-Aug-2019""",107000,742,20084,20118,1597,"""Petrol""",92,"""Auto""",2019,"""2""",22190,"""1,237 kg""","""Available for sale""","""Meticulously maintained and ke…","""Cosmic Blue Metallic, 18"" Weds…","""Mid-Sized Sedan""",72800
"""https://www.sgcarmart.com/used…",1487525,"""Mercedes-Benz B-Class B180""",10940,39463,"""25-Jan-2010""",156900,1290,27986,27986,1699,"""Petrol""",85,"""Auto""",2009,"""4""",15072,"""1,285 kg""","""Available for sale""","""Stock condition""","""In-car recorder and reverse ca…","""Hatchback""",41800
"""https://www.sgcarmart.com/used…",1470714,"""Mercedes-Benz GLB-Class GLB200…",23890,40989,"""16-Oct-2020""",70000,586,38946,46525,1332,"""Petrol""",120,"""Auto""",2020,"""2""",51187,"""1,610 kg""","""Available for sale""","""1.3l 4 cylinder inline 16 valv…","""AMG rims & bodykits, 10.25"" to…","""SUV""",131800
